In [ ]:
# Example 1: Basic ONNX Model Creation and Inspection
# Demonstrate creating and inspecting simple ONNX models

import onnx
import onnx.helper
import onnx.checker
import numpy as np

def create_simple_linear_model():
    """Create a simple linear model: Y = X * W + B"""
    
    # Define input and output
    X = onnx.helper.make_tensor_value_info('X', onnx.TensorProto.FLOAT, [1, 3])
    Y = onnx.helper.make_tensor_value_info('Y', onnx.TensorProto.FLOAT, [1, 1])
    
    # Create weight and bias parameters  
    W_vals = np.array([[0.5], [0.3], [0.2]], dtype=np.float32)
    W = onnx.helper.make_tensor('W', onnx.TensorProto.FLOAT, [3, 1], W_vals.flatten())
    
    B_vals = np.array([0.1], dtype=np.float32)
    B = onnx.helper.make_tensor('B', onnx.TensorProto.FLOAT, [1], B_vals)
    
    # Create MatMul node (X * W)
    matmul_node = onnx.helper.make_node(
        'MatMul',
        inputs=['X', 'W'],
        outputs=['XW']
    )
    
    # Create Add node (XW + B)
    add_node = onnx.helper.make_node(
        'Add',
        inputs=['XW', 'B'],
        outputs=['Y']
    )
    
    # Create the graph
    graph = onnx.helper.make_graph(
        nodes=[matmul_node, add_node],
        name='simple_linear',
        inputs=[X],
        outputs=[Y],
        initializer=[W, B]
    )
    
    # Create the model
    model = onnx.helper.make_model(graph)
    model.opset_import[0].version = 13
    
    return model

# Create and inspect a simple model
print("=== Creating Simple ONNX Linear Model ===")
model = create_simple_linear_model()

print(f"Model IR version: {model.ir_version}")
print(f"Producer name: {model.producer_name}")
print(f"Graph name: {model.graph.name}")
print(f"Number of nodes: {len(model.graph.node)}")
print(f"Number of inputs: {len(model.graph.input)}")
print(f"Number of outputs: {len(model.graph.output)}")
print(f"Number of initializers: {len(model.graph.initializer)}")

# Validate the model: a checker failure raises and fails the cell
onnx.checker.check_model(model)
print("✅ Model validation passed")
assert (len(model.graph.node), len(model.graph.input), len(model.graph.output), len(model.graph.initializer)) == (2, 1, 1, 2)

# Inspect nodes
print(f"\nModel nodes:")
for i, node in enumerate(model.graph.node):
    print(f"  Node {i}: {node.op_type} ({node.input} -> {node.output})")

print("Example 1: Basic ONNX model creation completed successfully!")

In [ ]:
# Example 2: ONNX Model Graph Manipulation and Composition
# Demonstrate graph manipulation and model composition techniques

import onnx
import onnx.helper
import onnx.compose
import numpy as np

def create_preprocessing_model():
    """Create a preprocessing model (normalization)"""
    
    # Input and output
    X = onnx.helper.make_tensor_value_info('raw_input', onnx.TensorProto.FLOAT, [1, 4])
    Y = onnx.helper.make_tensor_value_info('normalized', onnx.TensorProto.FLOAT, [1, 4])
    
    # Mean and std for normalization
    mean = onnx.helper.make_tensor('mean', onnx.TensorProto.FLOAT, [4], 
                                   np.array([0.5, 0.5, 0.5, 0.5], dtype=np.float32))
    std = onnx.helper.make_tensor('std', onnx.TensorProto.FLOAT, [4],
                                  np.array([0.2, 0.2, 0.2, 0.2], dtype=np.float32))
    
    # Normalize: (X - mean) / std
    sub_node = onnx.helper.make_node('Sub', ['raw_input', 'mean'], ['centered'])
    div_node = onnx.helper.make_node('Div', ['centered', 'std'], ['normalized'])
    
    graph = onnx.helper.make_graph(
        [sub_node, div_node],
        'preprocessing',
        [X], [Y],
        [mean, std]
    )
    
    model = onnx.helper.make_model(graph)
    model.opset_import[0].version = 13
    return model

def create_inference_model():
    """Create a simple inference model"""
    
    X = onnx.helper.make_tensor_value_info('normalized', onnx.TensorProto.FLOAT, [1, 4])
    Y = onnx.helper.make_tensor_value_info('prediction', onnx.TensorProto.FLOAT, [1, 2])
    
    # Weights
    W = onnx.helper.make_tensor('weights', onnx.TensorProto.FLOAT, [4, 2],
                                np.random.randn(4, 2).astype(np.float32).flatten())
    B = onnx.helper.make_tensor('bias', onnx.TensorProto.FLOAT, [2],
                                np.array([0.1, 0.2], dtype=np.float32))
    
    # MatMul + Add
    matmul_node = onnx.helper.make_node('MatMul', ['normalized', 'weights'], ['matmul_out'])
    add_node = onnx.helper.make_node('Add', ['matmul_out', 'bias'], ['prediction'])
    
    graph = onnx.helper.make_graph(
        [matmul_node, add_node],
        'inference',
        [X], [Y],
        [W, B]
    )
    
    model = onnx.helper.make_model(graph)
    model.opset_import[0].version = 13
    return model

print("=== ONNX Model Graph Manipulation and Composition ===")

# Create individual models
preprocessing_model = create_preprocessing_model()
inference_model = create_inference_model()

print(f"Preprocessing model: {len(preprocessing_model.graph.node)} nodes")
print(f"Inference model: {len(inference_model.graph.node)} nodes")

# Compose models using onnx.compose
print(f"\n=== Model Composition ===")
try:
    # Merge the two models - output of preprocessing becomes input of inference
    combined_model = onnx.compose.merge_models(
        preprocessing_model,
        inference_model,
        io_map=[('normalized', 'normalized')]  # Connect preprocessing output to inference input
    )
    
    print(f"Combined model: {len(combined_model.graph.node)} nodes")
    print(f"Combined inputs: {[inp.name for inp in combined_model.graph.input]}")
    print(f"Combined outputs: {[out.name for out in combined_model.graph.output]}")
    
    # Validate combined model
    onnx.checker.check_model(combined_model)
    print("✅ Combined model validated successfully")
    
except Exception as e:
    print(f"Model composition error: {e}")
    raise

# Demonstrate graph manipulation - add nodes to existing model
print(f"\n=== Graph Manipulation ===")
try:
    # Clone the inference model
    import copy
    modified_model = copy.deepcopy(inference_model)
    
    # Add a Sigmoid activation to the output
    sigmoid_node = onnx.helper.make_node(
        'Sigmoid',
        ['prediction'],
        ['sigmoid_out']
    )
    
    # Add node to graph
    modified_model.graph.node.extend([sigmoid_node])
    
    # Update output to use sigmoid output
    new_output = onnx.helper.make_tensor_value_info('sigmoid_out', onnx.TensorProto.FLOAT, [1, 2])
    modified_model.graph.output.pop()
    modified_model.graph.output.append(new_output)
    
    print(f"Original model nodes: {len(inference_model.graph.node)}")
    print(f"Modified model nodes: {len(modified_model.graph.node)}")
    print(f"Modified output: {modified_model.graph.output[0].name}")
    
    # Validate modified model
    onnx.checker.check_model(modified_model)
    print("✅ Modified model validated successfully")
    
except Exception as e:
    print(f"Graph manipulation error: {e}")
    raise

# Analyze node connections
print(f"\n=== Graph Analysis ===")
def analyze_graph(model, name):
    print(f"\n{name}:")
    print(f"  Nodes: {len(model.graph.node)}")
    print(f"  Initializers: {len(model.graph.initializer)}")
    
    # Build input-output map
    node_outputs = set()
    for node in model.graph.node:
        print(f"    {node.op_type}: {node.input} → {node.output}")
        node_outputs.update(node.output)
    
    # Check for unused initializers
    used_initializers = set()
    for node in model.graph.node:
        used_initializers.update(node.input)
    
    unused = set(init.name for init in model.graph.initializer) - used_initializers
    if unused:
        print(f"  Unused initializers: {unused}")

analyze_graph(preprocessing_model, "Preprocessing Model")
analyze_graph(inference_model, "Inference Model")
if 'combined_model' in locals():
    analyze_graph(combined_model, "Combined Model")

print("\nExample 2: ONNX graph manipulation and composition completed successfully!")

In [ ]:
# Example 3: ONNX Model Shape Inference and Type Checking  
# Demonstrate shape inference and type validation capabilities

import onnx
import onnx.helper
import onnx.shape_inference
import onnx.checker

def create_conv_model_with_shape_issues():
    """Create a CNN model to demonstrate shape inference"""
    
    # Input: batch_size=1, channels=3, height=32, width=32
    X = onnx.helper.make_tensor_value_info('input', onnx.TensorProto.FLOAT, [1, 3, 32, 32])
    
    # Conv layer weights: out_channels=16, in_channels=3, kernel_h=3, kernel_w=3
    conv_weights = onnx.helper.make_tensor(
        'conv_weights', onnx.TensorProto.FLOAT, [16, 3, 3, 3],
        np.random.randn(16, 3, 3, 3).astype(np.float32).flatten()
    )
    
    # Conv bias
    conv_bias = onnx.helper.make_tensor(
        'conv_bias', onnx.TensorProto.FLOAT, [16],
        np.random.randn(16).astype(np.float32)
    )
    
    # Create Conv node
    conv_node = onnx.helper.make_node(
        'Conv',
        inputs=['input', 'conv_weights', 'conv_bias'],
        outputs=['conv_output'],
        kernel_shape=[3, 3],
        pads=[1, 1, 1, 1],  # SAME padding
        strides=[1, 1]
    )
    
    # Create ReLU node
    relu_node = onnx.helper.make_node(
        'Relu',
        inputs=['conv_output'],
        outputs=['relu_output']
    )
    
    # Create Global Average Pool
    pool_node = onnx.helper.make_node(
        'GlobalAveragePool',
        inputs=['relu_output'],
        outputs=['pool_output']
    )
    
    # Reshape for fully connected layer
    reshape_node = onnx.helper.make_node(
        'Flatten',
        inputs=['pool_output'],
        outputs=['flattened']
    )
    
    # Output (without explicit shape - will be inferred)
    Y = onnx.helper.make_tensor_value_info('output', onnx.TensorProto.FLOAT, [None, None])
    
    # Final output node
    identity_node = onnx.helper.make_node(
        'Identity',
        inputs=['flattened'],
        outputs=['output']
    )
    
    graph = onnx.helper.make_graph(
        [conv_node, relu_node, pool_node, reshape_node, identity_node],
        'conv_model',
        [X], [Y],
        [conv_weights, conv_bias]
    )
    
    model = onnx.helper.make_model(graph)
    model.opset_import[0].version = 13
    return model

print("=== ONNX Shape Inference and Validation ===")

# Create model with undefined output shapes
model = create_conv_model_with_shape_issues()

print("Original model output info:")
for output in model.graph.output:
    shape = [d.dim_value if d.dim_value else "unknown" for d in output.type.tensor_type.shape.dim]
    print(f"  {output.name}: {shape}")

# Perform shape inference
print(f"\n=== Performing Shape Inference ===")
inferred_model = onnx.shape_inference.infer_shapes(model)

print("After shape inference:")
inferred_outputs = {}
for output in inferred_model.graph.output:
    shape = [d.dim_value if d.dim_value else f"dim_{d.dim_param}" for d in output.type.tensor_type.shape.dim]
    inferred_outputs[output.name] = shape
    print(f"  {output.name}: {shape}")

# Show intermediate tensor shapes too
print("\nIntermediate tensor shapes:")
intermediate_shapes = {}
for value_info in inferred_model.graph.value_info:
    if value_info.type.tensor_type.shape.dim:
        shape = [d.dim_value if d.dim_value else f"dim_{d.dim_param}" for d in value_info.type.tensor_type.shape.dim]
        intermediate_shapes[value_info.name] = shape
        print(f"  {value_info.name}: {shape}")

assert inferred_outputs == {'output': [1, 16]}, inferred_outputs
assert intermediate_shapes == {'conv_output': [1, 16, 32, 32], 'relu_output': [1, 16, 32, 32],
                               'pool_output': [1, 16, 1, 1], 'flattened': [1, 16]}, intermediate_shapes

# Model validation with detailed error checking
print(f"\n=== Model Validation ===")
# The model built above is valid: check_model raises on an invalid one, and that fails the cell
onnx.checker.check_model(model, full_check=True)
print("✅ Model structure validation passed")

# Additional type checking
for node in model.graph.node:
    print(f"Node {node.name or 'unnamed'} ({node.op_type}): {len(node.input)} inputs → {len(node.output)} outputs")

# Check for common issues
input_names = {inp.name for inp in model.graph.input}
initializer_names = {init.name for init in model.graph.initializer}

print(f"Model inputs: {input_names}")
print(f"Model initializers: {len(initializer_names)}")

# Validate that all node inputs are available
available_tensors = input_names | initializer_names
for node in model.graph.node:
    for input_name in node.input:
        assert input_name in available_tensors, f"Node {node.op_type} references undefined input: {input_name}"
        available_tensors.add(node.output[0] if node.output else "")

import onnx.version_converter
info = onnx.helper.make_tensor_value_info
opset_13 = [onnx.helper.make_opsetid('', 13)]
# Invalid model: Relu reads 'missing', which no input or earlier node defines
unsorted_model = onnx.helper.make_model(onnx.helper.make_graph(
    [onnx.helper.make_node('Relu', ['missing'], ['y'])], 'unsorted',
    [info('x', onnx.TensorProto.FLOAT, [2])], [info('y', onnx.TensorProto.FLOAT, [2])]), opset_imports=opset_13)
# Shape conflict: [2, 3] + [4, 5] cannot broadcast, and strict mode raises instead of skipping the node
conflict_model = onnx.helper.make_model(onnx.helper.make_graph(
    [onnx.helper.make_node('Add', ['a', 'b'], ['c'])], 'conflict',
    [info('a', onnx.TensorProto.FLOAT, [2, 3]), info('b', onnx.TensorProto.FLOAT, [4, 5])],
    [info('c', onnx.TensorProto.FLOAT, None)]), opset_imports=opset_13)
# The version converter does not support a sparse Constant
sparse = onnx.helper.make_sparse_tensor(onnx.helper.make_tensor('v', onnx.TensorProto.FLOAT, [2], [1.0, 2.0]),
                                        onnx.helper.make_tensor('i', onnx.TensorProto.INT64, [2], [0, 3]), [4])
sparse_model = onnx.helper.make_model(onnx.helper.make_graph(
    [onnx.helper.make_node('Constant', [], ['y'], sparse_value=sparse)], 'sparse', [],
    [info('y', onnx.TensorProto.FLOAT, [4])]), opset_imports=opset_13)

print(f"\n=== Typed errors ===")
caught = None
try:
    onnx.checker.check_model(unsorted_model)
except (RuntimeError, onnx.checker.ValidationError) as e:
    caught = e
assert type(caught) is onnx.checker.ValidationError, type(caught)
print(f"check_model raised {type(caught).__name__}, as expected")

caught = None
try:
    onnx.shape_inference.infer_shapes(conflict_model, strict_mode=True)
except (RuntimeError, onnx.shape_inference.InferenceError) as e:
    caught = e
assert type(caught) is onnx.shape_inference.InferenceError, type(caught)
print(f"infer_shapes raised {type(caught).__name__}, as expected")

caught = None
try:
    onnx.version_converter.convert_version(sparse_model, 12)
except (RuntimeError, onnx.version_converter.ConvertError) as e:
    caught = e
assert type(caught) is onnx.version_converter.ConvertError, type(caught)
print(f"convert_version raised {type(caught).__name__}, as expected")

print("Example 3: Shape inference and type checking completed successfully!")

In [ ]:
# Example 4: ONNX Model Serialization and Version Management
# Demonstrate model serialization, loading, and version compatibility

import onnx
import onnx.helper
import onnx.version_converter
import tempfile
import os

def create_versioned_model():
    """Create a model to test version compatibility"""
    
    # Simple model with common operations
    X = onnx.helper.make_tensor_value_info('input', onnx.TensorProto.FLOAT, [1, 10])
    Y = onnx.helper.make_tensor_value_info('output', onnx.TensorProto.FLOAT, [1, 5])
    
    # Weights and bias
    W = onnx.helper.make_tensor('weights', onnx.TensorProto.FLOAT, [10, 5], 
                                np.random.randn(50).astype(np.float32))
    B = onnx.helper.make_tensor('bias', onnx.TensorProto.FLOAT, [5], 
                                np.random.randn(5).astype(np.float32))
    
    # MatMul + Add + Sigmoid
    matmul_node = onnx.helper.make_node('MatMul', ['input', 'weights'], ['matmul_out'])
    add_node = onnx.helper.make_node('Add', ['matmul_out', 'bias'], ['add_out'])
    sigmoid_node = onnx.helper.make_node('Sigmoid', ['add_out'], ['output'])
    
    graph = onnx.helper.make_graph(
        [matmul_node, add_node, sigmoid_node],
        'versioned_model',
        [X], [Y], [W, B]
    )
    
    model = onnx.helper.make_model(graph)
    model.opset_import[0].version = 11  # Use older opset for testing
    model.producer_name = "test_producer"
    model.producer_version = "1.0"
    model.model_version = 1
    
    return model

print("=== ONNX Model Serialization and Version Management ===")

# Create model with specific version
model = create_versioned_model()
print(f"Original model opset version: {model.opset_import[0].version}")
print(f"Producer: {model.producer_name} v{model.producer_version}")
print(f"Model version: {model.model_version}")
print(f"IR version: {model.ir_version}")

# Serialize model to bytes
print(f"\n=== Model Serialization ===")
serialized_bytes = model.SerializeToString()
print(f"Model serialized to {len(serialized_bytes):,} bytes")

# Test round-trip serialization
deserialized_model = onnx.load_from_string(serialized_bytes)
print(f"Model deserialized successfully")
print(f"Deserialized opset version: {deserialized_model.opset_import[0].version}")

# Verify models are equivalent
assert deserialized_model == model, 'round trip changed the model'
assert deserialized_model.opset_import[0].version == 11, deserialized_model.opset_import[0].version
onnx.checker.check_model(deserialized_model)
print("✅ Round-trip serialization successful")

# Test file-based serialization
print(f"\n=== File-based Serialization ===")
with tempfile.NamedTemporaryFile(suffix='.onnx', delete=False) as tmp_file:
    model_path = tmp_file.name

try:
    # Save to file
    onnx.save(model, model_path)
    file_size = os.path.getsize(model_path)
    print(f"Model saved to file: {file_size:,} bytes")
    
    # Load from file
    loaded_model = onnx.load(model_path)
    print(f"Model loaded from file successfully")
    print(f"Loaded model nodes: {len(loaded_model.graph.node)}")
    # The file holds exactly the serialized model and loads back unchanged
    assert file_size == len(serialized_bytes), (file_size, len(serialized_bytes))
    assert loaded_model == model, 'file round trip changed the model'
finally:
    # Clean up whether or not the round trip held
    if os.path.exists(model_path):
        os.unlink(model_path)

# Version conversion
print(f"\n=== Version Conversion ===")
# Convert to newer opset version
newer_model = onnx.version_converter.convert_version(model, target_version=13)
print(f"Converted from opset {model.opset_import[0].version} to {newer_model.opset_import[0].version}")

# Validate converted model
onnx.checker.check_model(newer_model)
print("✅ Version conversion successful")

# Compare model sizes
original_size = len(model.SerializeToString())
converted_size = len(newer_model.SerializeToString())
print(f"Size change: {original_size:,} → {converted_size:,} bytes ({converted_size-original_size:+,})")
# The converter moves the opset and keeps the graph (MatMul/Add/Sigmoid unchanged)
assert newer_model.opset_import[0].version == 13, newer_model.opset_import[0].version
assert [n.op_type for n in newer_model.graph.node] == ['MatMul', 'Add', 'Sigmoid'], [n.op_type for n in newer_model.graph.node]

# Metadata and model information
print(f"\n=== Model Metadata ===")
metadata_props = []
if hasattr(model, 'metadata_props'):
    metadata_props = [(prop.key, prop.value) for prop in model.metadata_props]

print(f"Metadata properties: {len(metadata_props)}")
print(f"Model doc string: {model.doc_string or 'None'}")
print(f"Domain: {model.domain or 'Default'}")

# Add custom metadata
model.doc_string = "Example model for ONNX serialization testing"
model.domain = "example.com"

# Save model with metadata
print(f"Updated model doc string: {model.doc_string}")
print(f"Updated model domain: {model.domain}")

print("\nExample 4: Model serialization and version management completed successfully!")

In [ ]:
# Example 5: Production ONNX Model Pipeline and Runtime Preparation
# Demonstrate complete production workflow for ONNX models

import onnx
import onnx.helper
import onnx.numpy_helper
import numpy as np
from collections import defaultdict

class ONNXModelPipeline:
    """Production pipeline for ONNX model management"""
    
    def __init__(self):
        self.models = {}
        self.validation_results = {}
        self.optimization_stats = {}
    
    def create_classification_model(self, input_size=784, hidden_size=128, num_classes=10):
        """Create a complete classification model"""
        
        # Input layer
        X = onnx.helper.make_tensor_value_info('input', onnx.TensorProto.FLOAT, [None, input_size])
        Y = onnx.helper.make_tensor_value_info('output', onnx.TensorProto.FLOAT, [None, num_classes])
        
        # Layer 1: Dense + ReLU (using Xavier/Glorot initialization)
        xavier_std_1 = np.sqrt(2.0 / (input_size + hidden_size))
        W1 = onnx.helper.make_tensor('W1', onnx.TensorProto.FLOAT, [input_size, hidden_size],
                                     (np.random.randn(input_size, hidden_size) * xavier_std_1).astype(np.float32).flatten())
        B1 = onnx.helper.make_tensor('B1', onnx.TensorProto.FLOAT, [hidden_size],
                                     np.zeros(hidden_size, dtype=np.float32))
        
        dense1 = onnx.helper.make_node('MatMul', ['input', 'W1'], ['dense1_out'])
        add1 = onnx.helper.make_node('Add', ['dense1_out', 'B1'], ['add1_out'])
        relu1 = onnx.helper.make_node('Relu', ['add1_out'], ['relu1_out'])
        
        # Layer 2: Dense + Softmax
        xavier_std_2 = np.sqrt(2.0 / (hidden_size + num_classes))
        W2 = onnx.helper.make_tensor('W2', onnx.TensorProto.FLOAT, [hidden_size, num_classes],
                                     (np.random.randn(hidden_size, num_classes) * xavier_std_2).astype(np.float32).flatten())
        B2 = onnx.helper.make_tensor('B2', onnx.TensorProto.FLOAT, [num_classes],
                                     np.zeros(num_classes, dtype=np.float32))
        
        dense2 = onnx.helper.make_node('MatMul', ['relu1_out', 'W2'], ['dense2_out'])
        add2 = onnx.helper.make_node('Add', ['dense2_out', 'B2'], ['add2_out'])
        softmax = onnx.helper.make_node('Softmax', ['add2_out'], ['output'], axis=1)
        
        # Create graph
        graph = onnx.helper.make_graph(
            [dense1, add1, relu1, dense2, add2, softmax],
            'classification_model',
            [X], [Y], [W1, B1, W2, B2]
        )
        
        model = onnx.helper.make_model(graph)
        model.opset_import[0].version = 13
        model.producer_name = "ONNXModelPipeline"
        model.producer_version = "1.0"
        
        return model
    
    def validate_model_thoroughly(self, model, model_name):
        """Comprehensive model validation"""
        results = {
            'structure_valid': False,
            'shapes_inferred': False,
            'node_count': len(model.graph.node),
            'parameter_count': 0,
            'issues': []
        }
        
        try:
            # Structure validation
            onnx.checker.check_model(model, full_check=True)
            results['structure_valid'] = True
            
            # Shape inference
            inferred_model = onnx.shape_inference.infer_shapes(model)
            results['shapes_inferred'] = True
            
            # Count parameters
            total_params = 0
            for initializer in model.graph.initializer:
                param_count = 1
                for dim in initializer.dims:
                    param_count *= dim
                total_params += param_count
            results['parameter_count'] = total_params
            
            # Check for common issues
            node_types = defaultdict(int)
            for node in model.graph.node:
                node_types[node.op_type] += 1
            
            results['node_types'] = dict(node_types)
            
        except Exception as e:
            results['issues'].append(str(e))
        
        self.validation_results[model_name] = results
        return results
    
    def benchmark_model_performance(self, model, model_name):
        """Benchmark model for production readiness"""
        stats = {
            'model_size_bytes': len(model.SerializeToString()),
            'complexity_score': 0,
            'inference_ready': True
        }
        
        # Calculate complexity score (simple heuristic)
        complexity = 0
        for node in model.graph.node:
            if node.op_type in ['MatMul', 'Conv']:
                complexity += 10
            elif node.op_type in ['Add', 'Relu', 'Sigmoid']:
                complexity += 1
            else:
                complexity += 5
        
        stats['complexity_score'] = complexity
        stats['estimated_flops'] = complexity * 1000  # Rough estimate
        
        # Check if model is ready for deployment
        if stats['model_size_bytes'] > 100 * 1024 * 1024:  # 100MB
            stats['inference_ready'] = False
            stats['size_warning'] = "Model is very large for edge deployment"
        
        self.optimization_stats[model_name] = stats
        return stats
    
    def generate_deployment_report(self):
        """Generate comprehensive deployment report"""
        report = ["=== ONNX Model Deployment Report ===", ""]
        
        for model_name in self.models:
            validation = self.validation_results.get(model_name, {})
            performance = self.optimization_stats.get(model_name, {})
            
            report.extend([
                f"Model: {model_name}",
                f"  Structure Valid: {'✅' if validation.get('structure_valid') else '❌'}",
                f"  Shapes Inferred: {'✅' if validation.get('shapes_inferred') else '❌'}",
                f"  Node Count: {validation.get('node_count', 'N/A')}",
                f"  Parameters: {validation.get('parameter_count', 'N/A'):,}",
                f"  Model Size: {performance.get('model_size_bytes', 'N/A'):,} bytes",
                f"  Complexity Score: {performance.get('complexity_score', 'N/A')}",
                f"  Deployment Ready: {'✅' if performance.get('inference_ready') else '❌'}",
                ""
            ])
            
            if validation.get('issues'):
                report.append(f"  Issues: {'; '.join(validation['issues'])}")
                report.append("")
        
        return "\n".join(report)

# Run production pipeline
print("=== Production ONNX Model Pipeline ===")

pipeline = ONNXModelPipeline()

# Create and process multiple models
model_configs = [
    {"name": "small_classifier", "input_size": 128, "hidden_size": 64, "num_classes": 5},
    {"name": "medium_classifier", "input_size": 784, "hidden_size": 256, "num_classes": 10},
    {"name": "large_classifier", "input_size": 2048, "hidden_size": 512, "num_classes": 20}
]

for config in model_configs:
    print(f"\n--- Processing {config['name']} ---")
    
    # Create model
    model = pipeline.create_classification_model(**{k: v for k, v in config.items() if k != 'name'})
    pipeline.models[config['name']] = model
    
    # Validate model
    validation_results = pipeline.validate_model_thoroughly(model, config['name'])
    print(f"Validation: {'✅ PASSED' if validation_results['structure_valid'] else '❌ FAILED'}")
    print(f"Parameters: {validation_results['parameter_count']:,}")
    
    # Benchmark performance
    perf_results = pipeline.benchmark_model_performance(model, config['name'])
    print(f"Size: {perf_results['model_size_bytes']:,} bytes")
    print(f"Complexity: {perf_results['complexity_score']}")

# Generate final deployment report
print(f"\n{pipeline.generate_deployment_report()}")

# Summary statistics
total_params = sum(results['parameter_count'] for results in pipeline.validation_results.values())
total_size = sum(stats['model_size_bytes'] for stats in pipeline.optimization_stats.values())

print(f"Pipeline Summary:")
print(f"  Total models processed: {len(pipeline.models)}")
print(f"  Total parameters: {total_params:,}")
print(f"  Total size: {total_size:,} bytes ({total_size/1024/1024:.1f} MB)")

print("\nExample 5: Production ONNX model pipeline completed successfully!")